In [1]:
import xrayscatteringtools as xrst
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from matplotlib.colors import LogNorm
import warnings
xrst.enable_underscore_cleanup()

In [ ]:
###############################################
runNumbers = [] # <- this must be a list of int.
folders = xrst.get_data_paths(runNumbers) # Defaults to the info in config.yaml. You can overwrite this with strings, character arrays, or lists of either.
_phiOffset = 0 # Phi offset in radians, Either set this to -np.pi()/2 or 0 for fast analysis.
###############################################
# (1) keys_to_combine: some keys loaded for each shot & stored per shot 
# (2) keys_to_sum: some keys loaded per each run and added 
# (3) keys_to_check : check if some keys exist and have same values in all runs and load these keys 
_keys_to_combine = [
    # Azimuthal Average
    'jungfrau4M/azav_azav',
    # Photon energy
    'ebeam/photon_energy',
    # Timetool data
    'tt/FLTPOS', 'tt/AMPL', 'tt/FLTPOSFWHM',
     # Laser-Xray Timing. Use either time tool corrected (ttc) or not.
    # 'scan/lxt',
    'scan/lxt_ttc',
    # Upstream Diode
    'ipm_dg2/sum',
    # Gas detectors
    'gas_detector/f_11_ENRC', 'gas_detector/f_22_ENRC',
    # Light status
    'lightStatus/laser', 'lightStatus/xray',
]
_keys_to_sum = [
    'Sums/jungfrau4M_calib',
    # 'Sums/jungfrau4M_calib_thresADU1'
]
_keys_to_check = [
    'UserDataCfg/jungfrau4M/cmask', # Combined mask
    'UserDataCfg/jungfrau4M/azav__azav_q', # q bin centers
    'UserDataCfg/jungfrau4M/azav__azav_qbin', # q bin size
    'UserDataCfg/jungfrau4M/azav__azav_qbins', # q bin edges
    # These keys are typically not needed, but feel free to uncomment them.
    'UserDataCfg/jungfrau4M/azav__azav_idxq',
    'UserDataCfg/jungfrau4M/azav__azav_idxphi',
    'UserDataCfg/jungfrau4M/azav__azav_phiVec',
    # 'UserDataCfg/jungfrau4M/azav__azav_nphi',
    # 'UserDataCfg/jungfrau4M/azav__azav_matrix_q',
    # 'UserDataCfg/jungfrau4M/azav__azav_matrix_phi',
]
##### Load the data in #####
_data = xrst.combineRuns(runNumbers, folders, _keys_to_combine, _keys_to_sum, _keys_to_check, verbose=False)  # this is the function to load the data with defined keys
############################
# String for nice things
runNumbersRange = xrst.compress_ranges(runNumbers)
runType = xrst.get_config_for_runs(runNumbers[0],'samples','sample') # Default to information in the first run you load.
niceTitle = f"{xrst.get_config('expNumber')} : {'Run' if np.size(runNumbers)==1 else 'Runs'} {runNumbersRange} : {runType}"

# Jungfrau4M Data
azav = np.squeeze(_data['jungfrau4M/azav_azav'])
J4MSum = np.nansum(azav,axis=tuple(range(1, azav.ndim)))
q = _data['UserDataCfg/jungfrau4M/azav__azav_q'] # q bin centers
qbin = _data['UserDataCfg/jungfrau4M/azav__azav_qbin'] # q bin-size
qbins = _data['UserDataCfg/jungfrau4M/azav__azav_qbins'] # q bins edges
phibins = _data['UserDataCfg/jungfrau4M/azav__azav_phi'] + _phiOffset # phi bin centers
idxq = _data['UserDataCfg/jungfrau4M/azav__azav_idxq'].reshape(8,512,1024)
idxphi = _data['UserDataCfg/jungfrau4M/azav__azav_idxphi'].reshape(8,512,1024)
jungfrau_sum = _data['Sums/jungfrau4M_calib_thresADU1']   # Total Jungfrau detector counts with Thresholds added, summed in a run
cmask = _data['UserDataCfg/jungfrau4M/cmask'].astype(bool) # Mask for detector created 

# Scan and timetool information
scan = _data['scan/lxt_ttc']
# scan = data['scan/lxt']
time_bins = (np.unique(scan[:]))*10**12  # list of time bins in picoseconds
ttpos = _data['tt/FLTPOS']
ttampl = _data['tt/AMPL']
ttfwhm = _data['tt/FLTPOSFWHM']

# Event codes
laserOn = _data['lightStatus/laser'].astype(bool)  # laser on events
xrayOn = _data['lightStatus/xray'].astype(bool)  # xray on events
run_indicator = _data['run_indicator'] # run indicator for each shot

# X-ray beam diagnostics
dg2 = _data['ipm_dg2/sum']   # upstream diode x-ray intensity
pulse_energy = _data['gas_detector/f_11_ENRC']   # xray energy from gas detector (not calibrated to actual values)
photon_energy = xrst.get_config_for_runs(runNumbers[0],'photon_energy','energy')    # x-ray energy energy in eV
# Print total shots
_total_shots = len(run_indicator)
print("Total shots: ", _total_shots)

In [ ]:
###############################################
if True: # For plotting or not
###############################################
    _masked_sum = np.copy(jungfrau_sum)
    _masked_sum[~cmask.astype(bool)] = np.nan
    plt.figure(figsize=(10,8))
    pcm = xrst.plot_j4m(_masked_sum,vmin=0,vmax=np.nanpercentile(_masked_sum,99.5))
    plt.colorbar(pcm)
    plt.title('1 ADU Jungfrau4M Sum')
    plt.suptitle(niceTitle)
    plt.show()

In [ ]:
########## Different filter cutoffs##########
_J4M_cutoff = [0.4, 1];
_dg2_cutoff = [0.4, 1];
_pulse_energy_cutoff = [0.5, 1.5] # In mJ!!!
_tt_edgePos_cutoff = [0.1, 1]     
_tt_amp_cutoff = [0.3, 1]
_tt_width_cutoff = [0.1, 0.3]
_plot_display = 'log' # 'linear'
#############################################

# Precomputing the normalized values
_J4MSumNorm = J4MSum/np.nanmax(J4MSum);
_dg2Norm = dg2/np.nanmax(dg2);
_ttposNorm = ttpos/np.nanmax(ttpos)
_ttamplNorm = ttampl/np.nanmax(ttampl)
_ttfwhmNorm = ttfwhm/np.nanmax(ttfwhm)

plt.figure(figsize=[17,10]) 

plt.subplot(2,3,1)
plt.hist(_J4MSumNorm,bins=200,range=[0,np.nanmax(_J4MSumNorm)]);
plt.axvline(_J4M_cutoff[0],color='r',linestyle='--')
plt.axvline(_J4M_cutoff[1],color='r',linestyle='--')
plt.axvspan(_J4M_cutoff[0],_J4M_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('J4M Sum Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')
plt.legend()

plt.subplot(2,3,2)
plt.hist(_dg2Norm,bins=200,range=[0,np.nanmax(_dg2Norm)]);
plt.axvline(_dg2_cutoff[0],color='r',linestyle='--')
plt.axvline(_dg2_cutoff[1],color='r',linestyle='--')
plt.axvspan(_dg2_cutoff[0],_dg2_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('DG2 Sum Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')
plt.legend()

plt.subplot(2,3,3)
plt.hist(pulse_energy,bins=200,range=[0,np.nanmax(pulse_energy)]);
plt.axvline(_pulse_energy_cutoff[0],color='r',linestyle='--')
plt.axvline(_pulse_energy_cutoff[1],color='r',linestyle='--')
plt.axvspan(_pulse_energy_cutoff[0],_pulse_energy_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('Pulse Energy Histogram')
plt.xlabel('mJ')
plt.ylabel('Counts')
plt.legend()

plt.subplot(2,3,4)
plt.hist(_ttposNorm,bins=200,range=[0,np.nanmax(_ttposNorm)]);
plt.axvline(_tt_edgePos_cutoff[0],color='r',linestyle='--')
plt.axvline(_tt_edgePos_cutoff[1],color='r',linestyle='--')
plt.axvspan(_tt_edgePos_cutoff[0],_tt_edgePos_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('Timetool Edge Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')
plt.legend()

plt.subplot(2,3,5)
plt.hist(_ttamplNorm,bins=200,range=[0,np.nanmax(_ttamplNorm)]);
plt.axvline(_tt_amp_cutoff[0],color='r',linestyle='--')
plt.axvline(_tt_amp_cutoff[1],color='r',linestyle='--')
plt.axvspan(_tt_amp_cutoff[0],_tt_amp_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('Timetool Amplitude Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')
plt.legend()

plt.subplot(2,3,6)
plt.hist(_ttfwhmNorm,bins=200,range=[0,np.nanmax(_ttfwhmNorm)]);
plt.axvline(_tt_width_cutoff[0],color='r',linestyle='--')
plt.axvline(_tt_width_cutoff[1],color='r',linestyle='--')
plt.axvspan(_tt_width_cutoff[0],_tt_width_cutoff[1],color='r',alpha=0.2,label='"Good" Data')
plt.yscale(_plot_display)
plt.title('Timetool FWHM Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')
plt.legend()

plt.suptitle(niceTitle)
plt.show()
goodIdx = np.logical_and.reduce([
    _J4M_cutoff[0] <= _J4MSumNorm,
    _J4MSumNorm <= _J4M_cutoff[1],
    _dg2_cutoff[0] <= _dg2Norm,
    _dg2Norm <= _dg2_cutoff[1],
    _pulse_energy_cutoff[0] <= pulse_energy,
    pulse_energy <= _pulse_energy_cutoff[1],
    xrayOn
])
goodIdx_timetool = np.logical_and.reduce([
    _tt_edgePos_cutoff[0] <= _ttposNorm,
    _ttposNorm <= _tt_edgePos_cutoff[1],
    _tt_amp_cutoff[0] <= _ttamplNorm,
    _ttamplNorm <= _tt_amp_cutoff[1],
    _tt_width_cutoff[0] <= _ttfwhmNorm,
    _ttfwhmNorm <= _tt_width_cutoff[1],
    xrayOn, laserOn
])
# Displaying how much data was kept due to this filtering
_counts, _ = np.histogram(goodIdx.astype(int),[0,1,2]);
_counts_timetool, _ = np.histogram(goodIdx_timetool.astype(int),[0,1,2]);
print(f'goodIdx data represents {_counts[1]/np.sum(_counts)*100:.2f}% of the total shots. ({_counts[1]} out of {np.sum(_counts)}).')
print(f'goodIdx_timetool data represents {_counts_timetool[1]/np.sum(_counts_timetool)*100:.2f}% of the total shots. ({_counts_timetool[1]} out of {np.sum(_counts_timetool)}).')

In [ ]:
#####################################################################
# Define your bounds as dictionaries with slope 'm' and intercept 'b'
line_upper = {'m': 1.0, 'b': 0.15}
line_lower = {'m': 1.0, 'b': -0.15}
_bypass = False
#####################################################################

# Normalize axes to fraction of max
_dg2_subset = dg2[goodIdx]
_j4m_subset = J4MSum[goodIdx]

_dg2_min = np.nanmin(_dg2_subset)
_dg2_max = np.nanmax(_dg2_subset - _dg2_min)
_dg2_norm = (_dg2_subset - _dg2_min) / _dg2_max

_j4m_min = np.nanmin(_j4m_subset)
_j4m_max = np.nanmax(_j4m_subset - _j4m_min)
_j4m_norm = (_j4m_subset - _j4m_min) / _j4m_max

plt.figure(figsize=[12,5])

# 2D Histogram
plt.subplot(1,2,1)
plt.hist2d(_dg2_norm, _j4m_norm, bins=200, norm=LogNorm())
plt.colorbar()
plt.xlabel('DG2 (fraction of max)')
plt.ylabel('J4M (fraction of max)')
plt.title('J4M vs DG2 Histogram, Good Shots')

# Plot the arbitrary linear boundary lines
_x = np.linspace(0, 1, 500)
_y_lower_plot = line_lower['m'] * _x + line_lower['b']
_y_upper_plot = line_upper['m'] * _x + line_upper['b']

plt.plot(_x, _y_lower_plot, '--', color='r')
plt.plot(_x, _y_upper_plot, '--', color='r')

plt.xlim(0,1)
plt.ylim(0,1)

# 1D Histogram (Relative Position between lines)
plt.subplot(1,2,2)

# Wxpected Y values for every data point on both boundary lines
_y_bound_lower = line_lower['m'] * _dg2_norm + line_lower['b']
_y_bound_upper = line_upper['m'] * _dg2_norm + line_upper['b']

# Calculate relative position between the lines
# 1e-9 to prevent division by zero in case lines intersect
_relative_position = (_j4m_norm - _y_bound_lower) / (_y_bound_upper - _y_bound_lower + 1e-9)

# Plotting range is set slightly wider than [0, 1] to show the rejected data tails
plt.hist(_relative_position, bins=200, range=(-1.5, 2.5))
plt.axvline(0, color='r', linestyle='--')
plt.axvline(1, color='r', linestyle='--')
plt.axvspan(0, 1, color='r', alpha=0.2, label='"Good" Data')
plt.legend()
plt.xlabel('Relative Position (0=Lower Line, 1=Upper Line)')
plt.ylabel('Counts')
plt.title('Data Position Between Bounds')

plt.suptitle(niceTitle)
plt.tight_layout()
plt.show()

# make the boolean mask
if _bypass:
    
    _subset_pass = np.ones_like(_j4m_subset)
    print('Bypassing this filter')
else:
    _subset_pass = np.logical_and(
        _j4m_norm >= _y_bound_lower,
        _j4m_norm <= _y_bound_upper
    )
    _counts, _ = np.histogram(_subset_pass.astype(int),[0,1,2]);
    print(f'This filter retained {_counts[1]/np.sum(_counts)*100:.2f}% of the already filtered shots. ({_counts[1]} out of {np.sum(_counts)}).')
goodIdx_2 = np.zeros_like(goodIdx, dtype=bool)
goodIdx_2[goodIdx] = _subset_pass
# Displaying how much data was kept due to this filtering

In [ ]:
#############################################################################################################################################
do_timetool_correction = True
# Edit this as desired, will only have an effect if do_timetool_correction is True. Defaults to time_bins if list is empty. 
_new_time_bin_edges = np.arange(-2.5,2.5,0.03)
# Use the data in config.yaml or set your own, these should be in seconds, and can be scalar, sequence or ndarray
_tt_slopes = xrst.get_config_for_runs(runNumbers,'tt_calibration','slope')
_tt_intercepts = xrst.get_config_for_runs(runNumbers,'tt_calibration','intercept')
# Bool if you want
_plotting = True
#############################################################################################################################################

# Select bins, in ps
if do_timetool_correction and len(_new_time_bin_edges)!=0:
    time_bins_selected = _new_time_bin_edges
else:
    # Default to the time bins that were recorded
    time_bins_selected = time_bins

# Doing the timetool correction on a run by run basis
if do_timetool_correction:
    time_delay = xrst.calib.apply_timetool_correction(
        delays = scan,
        edge_positions = ttpos,
        slopes = _tt_slopes,
        intercepts = _tt_intercepts,
        run_indicator = run_indicator # Optional, if None, only one timetool calibration applied to all shots.
    )
    if _plotting:
        plt.figure(figsize=[12,5])
        plt.subplot(1,2,1)
        plt.hist(time_delay[goodIdx_timetool]*10**12,bins=200)
        plt.xlabel('Delay (ps)')
        plt.title('Timetool corrected delays, before binning')
        plt.ylabel('Counts')
        plt.subplot(1,2,2)
else:
    time_delay = scan
    
# Convert to ps, determione bin idx for each shot
binned_shot_idxs = np.digitize(time_delay*10**12,time_bins_selected)

if _plotting:
    plt.hist(time_delay[goodIdx_timetool]*10**12,bins=time_bins_selected)
    plt.xlabel('Delay (ps)')
    plt.title('Counts per selected time bin')
    plt.ylabel('Counts')
plt.suptitle(niceTitle)
plt.show()

## Define anisotropic functions from Eirik

In [ ]:
def leg_decomp_phi(S_tot,q_cent,phi_cent,weight):

    
    P_2 = 0.5*(3*np.cos(phi_cent)**2 - 1)
    
    S_0 = np.zeros(q_cent.shape)
    S_2 = np.zeros(q_cent.shape)
    S_avg = np.zeros(q_cent.shape)


    for i in range(0,len(q_cent)-1):

        S_avg[i] = np.nansum(S_tot[i]*weight[:,i])/np.nansum(weight[:,i])

        int_weight = weight[:,i].astype(int)
        S_points = np.repeat(S_tot[i],int_weight)
        P_points = np.repeat(P_2,int_weight)

        if np.sum(int_weight) > 0:
            log = np.isnan(S_points) == False
            S_2[i], S_0[i] = np.polyfit(P_points[log],S_points[log],1)
        

    return S_0, S_2, S_avg

def leg_decomp_theta_q(S_tot,q_cent,phi_cent,weight,E):

    k = 1 / xrst.keV2Angstroms(E)
    # k = 0.5067730716548338*E/1000
    
    S_0 = np.zeros(q_cent.shape)
    S_2 = np.zeros(q_cent.shape)
    S_avg = np.zeros(q_cent.shape)


    for i in range(0,len(q_cent)-1):

        theta_q = np.arccos(np.sqrt(1 - (q_cent[i]/2/k)**2)*np.cos(phi_cent))
        
        P_2 = 0.5*(3*np.cos(theta_q)**2 - 1)
        
        S_avg[i] = np.nansum(S_tot[i]*weight[:,i])/np.nansum(weight[:,i])

        int_weight = weight[:,i].astype(int)
        S_points = np.repeat(S_tot[i],int_weight)
        P_points = np.repeat(P_2,int_weight)

        if np.sum(int_weight) > 0:
            log = np.isnan(S_points) == False
            S_2[i], S_0[i] = np.polyfit(P_points[log],S_points[log],1)
        

    return S_0, S_2, S_avg

In [ ]:
if do_timetool_correction:
    # goodIdx_timetool already includes laserOn shots
    on_mask =  goodIdx_2 & goodIdx_timetool
else:
    # Just the good shots where the laser is on
    on_mask = goodIdx_2 & laserOn
# Timetool isn't needed for the off shots
off_mask = goodIdx_2 & ~laserOn

# Displaying how much data was kept due to this filtering
_counts, _ = np.histogram(on_mask.astype(int),[0,1,2]);
print(f'Combined filters reatined {_counts[1]/np.sum(laserOn)*100:.2f}% of the laser on shots. ({_counts[1]} out of {np.sum(laserOn)}).')
_counts, _ = np.histogram(off_mask.astype(int),[0,1,2]);
print(f'Combined filters reatined {_counts[1]/np.sum(~laserOn)*100:.2f}% of the laser off shots. ({_counts[1]} out of {np.sum(~laserOn)}).')

In [ ]:
##########################################################################################
_norm_method = 'dg2' # Supports 'dg2', 'none', 'q_range'
_q_range = [2.5,4] # This is only used if you select 'q-range' as the normalization method
##########################################################################################
if _norm_method == 'none':
    azav_norm = azav
elif _norm_method == 'dg2':
    with np.errstate(divide='ignore', invalid='ignore'):
        azav_norm = azav/dg2[:,np.newaxis]
elif _norm_method == 'q_range':
    # Determine valid q range idxs for filtereing the signal.
    _qidx_norm = (_q_range[0]<q) & (q<_q_range[1])
    with np.errstate(divide='ignore', invalid='ignore'):
        azav_norm = azav / np.nanmean(azav[:,_qidx_norm],axis=1)[:,np.newaxis]

In [ ]:
for i in np.unique(idxphi):
    for j in np.unique(idxq[idxq > 0]):
        bin_size[i,j] = np.sum((idxphi==i)*(idxq==j))

azav_xrays = azav*bin_size  # number of xrays for each shot
azav_laseroff = np.nanmean(np.transpose(azav_norm[off_mask]),axis=2)

# S_0_off = np.zeros_like(qbins)
# S_2_off = np.zeros_like(qbins)
# S_avg_off = np.zeros_like(qbins)

# Preallocation of arrays
S_0 = np.zeros((tbins,qbins.size)) 
S_2 = np.zeros((tbins,qbins.size))
S_avg = np.zeros((tbins,qbins.size)) 
onshots = np.zeros(tbins)

# 
for _j in range(0, tbins):
    timebin_on = np.where((binned_shot_idxs == _j) & on_mask)[0] #filter shots per timebin 
    onshots[_j] = timebin_on.size
    # Determine mean of shots in a specific time bin
    azav_tbin = np.nanmean(np.transpose(azav_norm[timebin_on]),axis=2)

S_0_off, S_2_off, S_avg_off = leg_decomp_phi(azav_laseroff,qbins,phibins,bin_size)

for _j in range(0,tbins):
    S_0[_j], S_2[_j], S_avg[_j] = leg_decomp_phi(azav_tbin[:,:,_j],qbins,phibins,bin_size)

In [ ]:
S_avg_diff = (S_avg - S_avg_off)/S_avg_off*100
S_0_diff = (S_0 - S_0_off)/S_0_off*100
S_2_diff = (S_2 - S_2_off)/S_0_off*100

t_plot = time_bins_selected*1000

In [ ]:
plt.pcolormesh(qbins,t_plot,S_avg_diff, cmap = 'RdBu_r')
plt.xlabel('q ($\mathrm{\AA}^{-1}$)',fontsize=14)
plt.ylabel('Time Delay (fs)',fontsize=14)
plt.title('Azimuthal average',fontsize=16)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlim([0.4,4])
plt.colorbar()
plt.clim(-4,4)
plt.show()

plt.pcolormesh(qbins,t_plot,S_0_diff, cmap = 'RdBu_r')
plt.xlabel('q ($\mathrm{\AA}^{-1}$)',fontsize=14)
plt.ylabel('Time Delay (fs)',fontsize=14)
plt.title('Isotropy',fontsize=16)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlim([0.4,4])
plt.colorbar()
plt.clim(-4,4)
plt.show()

plt.pcolormesh(qbins,t_plot,S_2_diff, cmap = 'RdBu_r')
plt.xlabel('q ($\mathrm{\AA}^{-1}$)',fontsize=14)
plt.ylabel('Time Delay (fs)',fontsize=14)
plt.title('Anisotropy (2nd order)',fontsize=16)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlim([0.4,4])
plt.colorbar()
plt.clim(-15,15)
plt.show()

plt.pcolormesh(qbins,t_plot,S_0_diff, cmap = 'RdBu_r')
plt.xlabel('q ($\mathrm{\AA}^{-1}$)',fontsize=14)
plt.ylabel('Time Delay (fs)',fontsize=14)
plt.title('Isotropy',fontsize=16)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlim([0.4,4])
plt.ylim([0,500])
plt.colorbar()
plt.clim(-3,3)
plt.show()

In [ ]:
for i in range(0,tbins):
    S_0[i], S_2[i], S_avg[i] = leg_decomp_theta_q(azav_tbin[:,:,i],qbins,phibins,bin_size,photon_energy)

In [ ]:
S_avg_diff = (S_avg - S_avg_off)/S_avg_off*100
S_0_diff = (S_0 - S_0_off)/S_0_off*100
S_2_diff = (S_2 - S_2_off)/S_0_off*100

t_plot = time_bins_selected*1000 + 650

plt.pcolormesh(qbins,t_plot,S_avg_diff, cmap = 'RdBu_r')
plt.xlabel('q ($\mathrm{\AA}^{-1}$)',fontsize=14)
plt.ylabel('Time Delay (fs)',fontsize=14)
plt.title('Azimuthal average',fontsize=16)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlim([0.4,4])
plt.colorbar()
plt.clim(-4,4)
plt.show()

plt.pcolormesh(qbins,t_plot,S_0_diff, cmap = 'RdBu_r')
plt.xlabel('q ($\mathrm{\AA}^{-1}$)',fontsize=14)
plt.ylabel('Time Delay (fs)',fontsize=14)
plt.title('Isotropy',fontsize=16)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlim([0.4,4])
plt.colorbar()
plt.clim(-4,4)
plt.show()

plt.pcolormesh(qbins,t_plot,S_2_diff, cmap = 'RdBu_r')
plt.xlabel('q ($\mathrm{\AA}^{-1}$)',fontsize=14)
plt.ylabel('Time Delay (fs)',fontsize=14)
plt.title('Anisotropy (2nd order)',fontsize=16)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlim([0.4,4])
plt.colorbar()
plt.clim(-15,15)
plt.show()

plt.pcolormesh(qbins,t_plot,S_0_diff, cmap = 'RdBu_r')
plt.xlabel('q ($\mathrm{\AA}^{-1}$)',fontsize=14)
plt.ylabel('Time Delay (fs)',fontsize=14)
plt.title('Isotropy',fontsize=16)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlim([0.4,4])
plt.ylim([0,500])
plt.colorbar()
plt.clim(-3,3)
plt.show()